In [2]:
import torch
import numpy as np
import pandas as pd
import wandb

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Test tensor:", torch.randn(2, 3))

PyTorch version: 2.13.0
CUDA available: False
Test tensor: tensor([[-1.8211, -1.4665, -0.0828],
        [-0.9678,  0.3029,  0.5424]])


In [3]:
#TEST PER VEDERE SE MPS GIRA SU QUESTO MACBOOK
import torch
import torch.nn as nn
import time


def benchmark_avgpool(device, n_runs=10000):
    print(f"Testing device: {device}")

    # Simuliamo un batch abbastanza grande
    x = torch.randn(64, 336, 7).to(device)

    pool = nn.AvgPool1d(
        kernel_size=25,
        stride=1,
        padding=0
    ).to(device)

    # AvgPool1d vuole shape [batch, channels, seq_len]
    x = x.permute(0, 2, 1)

    # Warm-up: qualche giro iniziale non misurato
    for _ in range(10):
        out = pool(x)

    # Su MPS serve sincronizzare prima di misurare
    if device == "mps":
        torch.mps.synchronize()

    start = time.time()

    for _ in range(n_runs):
        out = pool(x)

    # Su MPS serve sincronizzare anche dopo
    if device == "mps":
        torch.mps.synchronize()

    end = time.time()

    total_time = end - start
    avg_time = total_time / n_runs

    print(f"Total time: {total_time:.4f} seconds")
    print(f"Average time per run: {avg_time:.6f} seconds")
    print(f"Output shape: {out.shape}")
    print()


benchmark_avgpool("cpu")

if torch.backends.mps.is_available():
    benchmark_avgpool("mps")
else:
    print("MPS not available")

Testing device: cpu
Total time: 3.6224 seconds
Average time per run: 0.000362 seconds
Output shape: torch.Size([64, 7, 312])

Testing device: mps
Total time: 0.5738 seconds
Average time per run: 0.000057 seconds
Output shape: torch.Size([64, 7, 312])



In [4]:
### TESTING decomposition.py
import sys
import os

# Add the project root to Python path
sys.path.append(os.path.abspath(".."))

import torch
from layers.decomposition import SeriesDecomp

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Device:", device)

# Fake time series: [batch_size, seq_len, channels]
x = torch.randn(4, 96, 7).to(device)

# Autoformer usually uses moving_avg = 25
decomp = SeriesDecomp(kernel_size=25).to(device)
seasonal, trend = decomp(x)

print("Input shape:   ", x.shape)
print("Seasonal shape:", seasonal.shape)
print("Trend shape:   ", trend.shape)

# Check reconstruction: questa media mobile dovrebbe essere invertibile, quindi la somma di seasonal e trend dovrebbe essere uguale all'input originale
reconstruction_error = torch.mean(torch.abs((seasonal + trend) - x))

print("Reconstruction error:", reconstruction_error.item())
print(x[0, :5, 0])  # Mostra i primi 5 valori della prima serie temporale del batch
print(x[0, :5, 6])  # Mostra i primi 5 valori della settima e ultima serie temporale del batch

Device: mps
Input shape:    torch.Size([4, 96, 7])
Seasonal shape: torch.Size([4, 96, 7])
Trend shape:    torch.Size([4, 96, 7])
Reconstruction error: 6.0260303413883776e-09
tensor([ 1.1184,  0.8549,  0.3294, -0.0773,  0.4332], device='mps:0')
tensor([-1.3054, -1.3326,  0.2953,  0.2742, -0.0935], device='mps:0')


In [5]:
# TESTING embedding.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from layers.embedding import DataEmbeddingWithoutPos

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake input time series
# [batch_size, seq_len, number_of_variables]
x = torch.randn(4, 96, 7).to(device)

# Fake time features
# For freq="t", we use 5 temporal features
x_mark = torch.randn(4, 96, 5).to(device)

embedding = DataEmbeddingWithoutPos(
    c_in=7,
    d_model=512,
    freq="t",
    dropout=0.05
).to(device)

out = embedding(x, x_mark)

print("x shape:      ", x.shape)
print("x_mark shape: ", x_mark.shape)
print("output shape: ", out.shape)

Device: mps
x shape:       torch.Size([4, 96, 7])
x_mark shape:  torch.Size([4, 96, 5])
output shape:  torch.Size([4, 96, 512])


In [6]:
#TESTING autocorrelation.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from layers.autocorrelation import AutoCorrelation


# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake tensors already split into heads
# Shape: [batch_size, seq_len, n_heads, d_head]
B = 4
L = 96
H = 8
E = 64

queries = torch.randn(B, L, H, E).to(device)
keys = torch.randn(B, L, H, E).to(device)
values = torch.randn(B, L, H, E).to(device)

autocorr = AutoCorrelation(c=1).to(device)

out = autocorr(queries, keys, values)

print("queries shape:", queries.shape)
print("keys shape:   ", keys.shape)
print("values shape: ", values.shape)
print("output shape: ", out.shape)

Device: mps
queries shape: torch.Size([4, 96, 8, 64])
keys shape:    torch.Size([4, 96, 8, 64])
values shape:  torch.Size([4, 96, 8, 64])
output shape:  torch.Size([4, 96, 8, 64])


In [7]:
# TESTING autocorrelation.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake input tensors in normal model format
# Shape: [batch_size, seq_len, d_model]
B = 4
L = 96
d_model = 512
n_heads = 8

x = torch.randn(B, L, d_model).to(device)

autocorr = AutoCorrelation(c=1)
autocorr_layer = AutoCorrelationLayer(
    autocorrelation=autocorr,
    d_model=d_model,
    n_heads=n_heads
).to(device)

out = autocorr_layer(x, x, x)

print("input shape: ", x.shape)
print("output shape:", out.shape)

Device: mps
input shape:  torch.Size([4, 96, 512])
output shape: torch.Size([4, 96, 512])


In [8]:
#### TESTING encoderLayer.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch

from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer
from layers.encoder import EncoderLayer


# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake input tensor
# Shape: [batch_size, seq_len, d_model]
B = 4
L = 96
d_model = 512
n_heads = 8
d_ff = 2048
moving_avg = 25
c = 1
dropout = 0.1

x = torch.randn(B, L, d_model).to(device)

# Build AutoCorrelation layer
autocorrelation = AutoCorrelation(c=c)

autocorrelation_layer = AutoCorrelationLayer(
    autocorrelation=autocorrelation,
    d_model=d_model,
    n_heads=n_heads
)

# Build EncoderLayer
encoder_layer = EncoderLayer(
    autocorrelation_layer=autocorrelation_layer,
    d_model=d_model,
    d_ff=d_ff,
    moving_avg=moving_avg,
    dropout=dropout
).to(device)

# Forward pass
out = encoder_layer(x)

print("input shape: ", x.shape)
print("output shape:", out.shape)

Device: mps
input shape:  torch.Size([4, 96, 512])
output shape: torch.Size([4, 96, 512])


In [9]:
#### TESTING decoderLayer.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch

from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer
from layers.decoder import DecoderLayer

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake dimensions
B = 4
enc_len = 96
dec_len = 144      # label_len + pred_len, for example 48 + 96
d_model = 512
n_heads = 8
d_ff = 2048
moving_avg = 25
c_out = 7
c = 1
dropout = 0.1

# Fake decoder seasonal input and encoder output
x = torch.randn(B, dec_len, d_model).to(device)
cross = torch.randn(B, enc_len, d_model).to(device)

# Self Auto-Correlation layer for decoder
self_autocorrelation = AutoCorrelation(c=c)
self_attention_layer = AutoCorrelationLayer(
    autocorrelation=self_autocorrelation,
    d_model=d_model,
    n_heads=n_heads
)

# Cross Auto-Correlation layer
cross_autocorrelation = AutoCorrelation(c=c)
cross_attention_layer = AutoCorrelationLayer(
    autocorrelation=cross_autocorrelation,
    d_model=d_model,
    n_heads=n_heads
)

decoder_layer = DecoderLayer(
    self_attention_layer=self_attention_layer,
    cross_attention_layer=cross_attention_layer,
    d_model=d_model,
    c_out=c_out,
    d_ff=d_ff,
    moving_avg=moving_avg,
    dropout=dropout
).to(device)

out, residual_trend = decoder_layer(x, cross)

print("decoder input shape:   ", x.shape)
print("encoder output shape:  ", cross.shape)
print("decoder output shape:  ", out.shape)
print("residual trend shape:  ", residual_trend.shape)

Device: mps
decoder input shape:    torch.Size([4, 144, 512])
encoder output shape:   torch.Size([4, 96, 512])
decoder output shape:   torch.Size([4, 144, 512])
residual trend shape:   torch.Size([4, 144, 7])


In [10]:
### TESTING decoder.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn

from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer
from layers.decoder import DecoderLayer, Decoder

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake dimensions
B = 4
enc_len = 96
dec_len = 144      # label_len + pred_len = 48 + 96
d_model = 512
n_heads = 8
d_ff = 2048
moving_avg = 25
c_out = 7
c = 1
dropout = 0.1
dec_layers = 1

# Fake inputs
x = torch.randn(B, dec_len, d_model).to(device)      # seasonal decoder input
cross = torch.randn(B, enc_len, d_model).to(device)  # encoder output
trend = torch.randn(B, dec_len, c_out).to(device)    # initial trend

decoder_layers = []

for _ in range(dec_layers):
    self_autocorrelation = AutoCorrelation(c=c)
    self_attention_layer = AutoCorrelationLayer(
        autocorrelation=self_autocorrelation,
        d_model=d_model,
        n_heads=n_heads
    )

    cross_autocorrelation = AutoCorrelation(c=c)
    cross_attention_layer = AutoCorrelationLayer(
        autocorrelation=cross_autocorrelation,
        d_model=d_model,
        n_heads=n_heads
    )

    decoder_layer = DecoderLayer(
        self_attention_layer=self_attention_layer,
        cross_attention_layer=cross_attention_layer,
        d_model=d_model,
        c_out=c_out,
        d_ff=d_ff,
        moving_avg=moving_avg,
        dropout=dropout
    )

    decoder_layers.append(decoder_layer)

decoder = Decoder(
    decoder_layers=decoder_layers,
    norm_layer=nn.LayerNorm(d_model),
    projection=None
).to(device)

seasonal_out, trend_out = decoder(x, cross, trend)

print("decoder input seasonal shape:", x.shape)
print("encoder output shape:        ", cross.shape)
print("initial trend shape:         ", trend.shape)
print("seasonal output shape:       ", seasonal_out.shape)
print("trend output shape:          ", trend_out.shape)

Device: mps
decoder input seasonal shape: torch.Size([4, 144, 512])
encoder output shape:         torch.Size([4, 96, 512])
initial trend shape:          torch.Size([4, 144, 7])
seasonal output shape:        torch.Size([4, 144, 512])
trend output shape:           torch.Size([4, 144, 7])


In [11]:
#### TESTING autoformer.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from models.autoformer import Autoformer



# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

config = {
    "seq_len": 96,
    "label_len": 48,
    "pred_len": 96,

    "enc_in": 7,
    "dec_in": 7,
    "c_out": 7,

    "d_model": 512,
    "n_heads": 8,
    "d_ff": 2048,
    "enc_layers": 2,
    "dec_layers": 1,
    "moving_avg": 25,
    "c": 1,
    "dropout": 0.05,
    "freq": "t",
}

B = 4
seq_len = config["seq_len"]
label_len = config["label_len"]
pred_len = config["pred_len"]
dec_len = label_len + pred_len

enc_in = config["enc_in"]
dec_in = config["dec_in"]

# For freq="t", our TimeFeatureEmbedding expects 5 time features
time_features = 5

x_enc = torch.randn(B, seq_len, enc_in).to(device)
x_mark_enc = torch.randn(B, seq_len, time_features).to(device)

# x_dec is currently kept only for API consistency in our Autoformer forward
x_dec = torch.randn(B, dec_len, dec_in).to(device)
x_mark_dec = torch.randn(B, dec_len, time_features).to(device)

model = Autoformer(config).to(device)

out = model(
    x_enc=x_enc,
    x_mark_enc=x_mark_enc,
    x_dec=x_dec,
    x_mark_dec=x_mark_dec
)

print("x_enc shape:      ", x_enc.shape)
print("x_mark_enc shape: ", x_mark_enc.shape)
print("x_dec shape:      ", x_dec.shape)
print("x_mark_dec shape: ", x_mark_dec.shape)
print("output shape:     ", out.shape)


Device: mps
x_enc shape:       torch.Size([4, 96, 7])
x_mark_enc shape:  torch.Size([4, 96, 5])
x_dec shape:       torch.Size([4, 144, 7])
x_mark_dec shape:  torch.Size([4, 144, 5])
output shape:      torch.Size([4, 96, 7])


In [17]:
### TESTING dataset
import pandas as pd

df = pd.read_csv("../data/raw/ETTm2.csv")

print(df.shape)
print(df.head())
print(df.columns)

(69680, 8)
                  date       HUFL    HULL       MUFL   MULL   LUFL   LULL  \
0  2016-07-01 00:00:00  41.130001  12.481  36.535999  9.355  4.424  1.311   
1  2016-07-01 00:15:00  39.622002  11.309  35.543999  8.551  3.209  1.258   
2  2016-07-01 00:30:00  38.868000  10.555  34.365002  7.586  4.435  1.258   
3  2016-07-01 00:45:00  35.518002   9.214  32.569000  8.712  4.435  1.215   
4  2016-07-01 01:00:00  37.528000  10.136  33.936001  7.532  4.435  1.215   

          OT  
0  38.661999  
1  38.223000  
2  37.344002  
3  37.124001  
4  37.124001  
Index(['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT'], dtype='str')


In [19]:
import sys
import os

sys.path.append("..")

from data_provider.data_loader import ETTDataset

train_dataset = ETTDataset(
    data_path="../data/raw/ETTm2.csv",
    flag="train",
    seq_len=96,
    label_len=48,
    pred_len=96,
    features="M",
    target="OT",
    scale=True
)

print("Number of training windows:", len(train_dataset))




Number of training windows: 48585


In [28]:
sample = train_dataset[0]

x_enc, x_mark_enc, x_dec, x_mark_dec, y = sample

print("x_enc shape:", x_enc.shape)
print("x_mark_enc shape:", x_mark_enc.shape)
print("x_dec shape:", x_dec.shape)
print("x_mark_dec shape:", x_mark_dec.shape)
print("y shape:", y.shape)

x_enc shape: torch.Size([96, 7])
x_mark_enc shape: torch.Size([96, 5])
x_dec shape: torch.Size([144, 7])
x_mark_dec shape: torch.Size([144, 5])
y shape: torch.Size([96, 7])


In [ ]:
### TESTING data_loader.py
from data_provider.data_loader import get_data_loader

train_dataset, train_loader = get_data_loader(
    data_path="../data/raw/ETTm2.csv",
    flag="train",
    seq_len=96,
    label_len=48,
    pred_len=96,
    features="M",
    target="OT",
    batch_size=32,
    shuffle=True,
    scale=True
)

batch = next(iter(train_loader)) ### VUOL DIRE PRENDIMI IL PRIMO BATCH PRODOTTO DAL DATALOADER

x_enc, x_mark_enc, x_dec, x_mark_dec, y = batch

print("x_enc shape:", x_enc.shape)
print("x_mark_enc shape:", x_mark_enc.shape)
print("x_dec shape:", x_dec.shape)
print("x_mark_dec shape:", x_mark_dec.shape)
print("y shape:", y.shape)

x_enc shape: torch.Size([32, 96, 7])
x_mark_enc shape: torch.Size([32, 96, 5])
x_dec shape: torch.Size([32, 144, 7])
x_mark_dec shape: torch.Size([32, 144, 5])
y shape: torch.Size([32, 96, 7])
